# R515B Soğutucusu için STGP ve EF Tabanlı Özellik Zenginleştirme ve Regresyon Performans Analizi

Bu çalışma, R515B termodinamik soğutucu akışkanının doymuş ve kızgın fazlarındaki veri setlerinde, evrimsel özellik üretiminin (STGP ve EF) regresyon performansına ve termodinamik uyuma (monotonluk) etkisini inceler.

**Senaryolar**

| Senaryo | Girdi uzayı |
|---|---|
| `Base` | Ham girdiler |
| `STGP` | Ham girdiler + STGP ile üretilen öznitelikler |
| `EF` | Ham girdiler + EF (Evolutionary Forest) ile üretilen öznitelikler |
| `HYBRID` | Ham girdiler + STGP + EF öznitelikleri |
| `Domain_Knowledge` *(yalnızca doymuş faz, ayrı karşılaştırma)* | Ham girdi + 1/T + ln(T) |

`Domain_Knowledge` senaryosu, kazancın evrimsel özniteliklerden mi yoksa yalnızca termodinamik dönüşümlerden mi geldiğini ayırt etmek için eklenmiştir. STGP, EF ve HYBRID öznitelikleri yalnızca ham girdilerden üretilir.

**Yöntem**
- **9 farklı regresyon algoritması** (AdaBoost, CatBoost, DART, ET, GBDT, KNN, LightGBM, RF, XGBoost) **5-Fold Çapraz Doğrulama** ile değerlendirilir.
- **Veri sızıntısı önlemi:** Ölçekleyici, STGP, EF ve regresörler her katta **yalnızca o katın eğitim verisiyle** fit edilir. Test verisi ve sentetik ızgara yalnızca `transform`/`predict` görür. Tüm veriyle fit edilen herhangi bir öznitelik veya model sonuçlarda kullanılmaz.
- Üretilen öznitelikler "Evrensel Permutation Importance" ile (test katı üzerinde) değerlendirilir.
- Modellerin termodinamik tutarlılığı, monotonluk ihlal oranı ile sentetik ızgara üzerinde ölçülür; oranlar katlar üzerinden ortalanır. Yöntem fizik bilgili (PIML) olarak nitelendirilmez; ihlal oranları bir uyum ölçütüdür ve hibrit senaryoda artabilir.

*Tüm analizler tek bir K-Fold döngüsünde gerçekleştirilir ve sonuçlar `Results` klasörüne çok sekmeli Excel formatında kaydedilir.*

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from Functions2 import (
    SATURATED_INPUTS, SATURATED_OUTPUTS,
    SUPERHEATED_INPUTS, SUPERHEATED_OUTPUTS,
    run_unified_analysis, summarize_scenarios
)

2026-10-05 14:10:36,665 - INFO - Loading faiss with AVX2 support.
2026-10-05 14:10:36,682 - INFO - Successfully loaded faiss with AVX2 support.


In [2]:
# Uyarı: Kendi dizininize ve dosya adınıza göre veri okuma kodunu güncelleyebilirsiniz.
# Örnek okuma kodları:
df_doymus = pd.read_csv("R515B_Ozellikler\R515B_Doymus_Ozellikleri.csv") 
df_kizgin = pd.read_csv("R515B_Ozellikler\R515B_Kizgin_Buhar.csv") 

print("Doymuş Faz Veri Boyutu:", df_doymus.shape)
print("Kızgın Faz Veri Boyutu:", df_kizgin.shape)

Doymuş Faz Veri Boyutu: (136, 8)
Kızgın Faz Veri Boyutu: (578, 5)


In [3]:
# DOYMUŞ FAZ ANALİZİ (Base, STGP, EF, HYBRID + Domain_Knowledge)
sat_metrics, sat_importances, sat_uyum = run_unified_analysis(
    df=df_doymus,
    input_cols=SATURATED_INPUTS,
    outputs_list=SATURATED_OUTPUTS,
    dataset_name="Doymus_Faz"
)

print("\n--- Doymuş Faz: Senaryo Bazında Ortalama Skorlar ---")
display(summarize_scenarios(sat_metrics, sat_uyum))

print("\n--- Doymuş Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---")
display(sat_uyum.sort_values(by=['Ihlal_Yuzdesi_(%)', 'Target']).head(15))


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DOYMUS_FAZ - SIZINTISIZ K-FOLD VE TERMODİNAMİK UYUM ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

[FAZ 1/3] K-Fold dongusu: STGP-EF her katta yalnizca egitim verisiyle fit ediliyor...
  -> Hedef: P(çıktı)
  -> Hedef: v sıvı (çıktı)
  -> Hedef: v buhar (çıktı)
  -> Hedef: h sıvı (çıktı)
  -> Hedef: h buhar (çıktı)
  -> Hedef: s sıvı (çıktı)
  -> Hedef: s buhar (çıktı)

[FAZ 2/3] Fold sonuclari ortalaniyor...

[FAZ 3/3] Tum sonuclar Excel dosyasina kaydediliyor...
-> Çıktı Dosyası Başarıyla Oluşturuldu: Results\Doymus_Faz_Nihai_Sonuclar.xlsx

--- Doymuş Faz: Senaryo Bazında Ortalama Skorlar ---


,Test_R2,Test_MAPE,Ort_Ihlal_Yuzdesi_(%)
Scenario,,,
Base,-0.9094,0.0442,1.8070
STGP,-0.9018,0.0445,3.1134
EF,-0.8932,0.0419,3.2517
HYBRID,-0.8935,0.0420,3.7551
Domain_Knowledge,-0.9111,0.0428,2.1904



--- Doymuş Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---


,Target,Algoritma,Scenario,Test_Tipi,Fiziksel_Ihlal_Sayisi,Ihlal_Yuzdesi_(%)
0,P(çıktı),AdaBoost,Base,T degisiyor (Doymus),0.0000,0.0000
1,P(çıktı),AdaBoost,STGP,T degisiyor (Doymus),0.0000,0.0000
2,P(çıktı),AdaBoost,EF,T degisiyor (Doymus),0.0000,0.0000
3,P(çıktı),AdaBoost,HYBRID,T degisiyor (Doymus),0.0000,0.0000
4,P(çıktı),AdaBoost,Domain_Knowledge,T degisiyor (Doymus),0.0000,0.0000
10,P(çıktı),DART,Base,T degisiyor (Doymus),0.0000,0.0000
11,P(çıktı),DART,STGP,T degisiyor (Doymus),0.0000,0.0000
14,P(çıktı),DART,Domain_Knowledge,T degisiyor (Doymus),0.0000,0.0000
15,P(çıktı),ET,Base,T degisiyor (Doymus),0.0000,0.0000
19,P(çıktı),ET,Domain_Knowledge,T degisiyor (Doymus),0.0000,0.0000


In [4]:
# KIZGIN FAZ ANALİZİ (Base, STGP, EF, HYBRID)
sup_metrics, sup_importances, sup_uyum = run_unified_analysis(
    df=df_kizgin,
    input_cols=SUPERHEATED_INPUTS,
    outputs_list=SUPERHEATED_OUTPUTS,
    dataset_name="Kizgin_Faz"
)

print("\n--- Kızgın Faz: Senaryo Bazında Ortalama Skorlar ---")
display(summarize_scenarios(sup_metrics, sup_uyum))

print("\n--- Kızgın Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---")
display(sup_uyum.sort_values(by=['Ihlal_Yuzdesi_(%)', 'Target']).head(15))


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  KIZGIN_FAZ - SIZINTISIZ K-FOLD VE TERMODİNAMİK UYUM ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

[FAZ 1/3] K-Fold dongusu: STGP-EF her katta yalnizca egitim verisiyle fit ediliyor...
  -> Hedef: v (çıktı)
  -> Hedef: h (çıktı)
  -> Hedef: s (çıktı)

[FAZ 2/3] Fold sonuclari ortalaniyor...

[FAZ 3/3] Tum sonuclar Excel dosyasina kaydediliyor...
-> Çıktı Dosyası Başarıyla Oluşturuldu: Results\Kizgin_Faz_Nihai_Sonuclar.xlsx

--- Kızgın Faz: Senaryo Bazında Ortalama Skorlar ---


,Test_R2,Test_MAPE,Ort_Ihlal_Yuzdesi_(%)
Scenario,,,
Base,0.9160,0.0351,0.1428
STGP,0.9291,0.0368,1.5141
EF,0.9272,0.0327,1.4227
HYBRID,0.9303,0.0334,1.5968



--- Kızgın Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---


,Target,Algoritma,Scenario,Test_Tipi,Fiziksel_Ihlal_Sayisi,Ihlal_Yuzdesi_(%)
0,h (çıktı),AdaBoost,Base,"Sabit P=1.5, T degisiyor",0.0000,0.0000
1,h (çıktı),AdaBoost,Base,"Sabit T=40.0, P degisiyor",0.0000,0.0000
2,h (çıktı),AdaBoost,STGP,"Sabit P=1.5, T degisiyor",0.0000,0.0000
4,h (çıktı),AdaBoost,EF,"Sabit P=1.5, T degisiyor",0.0000,0.0000
6,h (çıktı),AdaBoost,HYBRID,"Sabit P=1.5, T degisiyor",0.0000,0.0000
8,h (çıktı),CatBoost,Base,"Sabit P=1.5, T degisiyor",0.0000,0.0000
9,h (çıktı),CatBoost,Base,"Sabit T=40.0, P degisiyor",0.0000,0.0000
16,h (çıktı),DART,Base,"Sabit P=1.5, T degisiyor",0.0000,0.0000
17,h (çıktı),DART,Base,"Sabit T=40.0, P degisiyor",0.0000,0.0000
19,h (çıktı),DART,STGP,"Sabit T=40.0, P degisiyor",0.0000,0.0000
